# Test des fonctions backend sur le dataset Parcoursup

Ce notebook vise à tester et illustrer l’utilisation des fonctions du backend (`src/backend/processing.py`) pour charger, nettoyer et filtrer le dataset Parcoursup.

## Objectif

- Charger le fichier brut depuis `data/raw/`
- Appliquer les fonctions de traitement du backend
- Inspecter rapidement le résultat (colonnes, nombre de lignes)

In [1]:
import os

os.getcwd()

'/Users/maelioviau-vonfelt/Desktop/Masters/Epitech Data/Cours/M2/Mémoires/Consulting - StudAI/etudIA/notebooks'

In [2]:
# Vérifier que Python cherche et pointe vers les bons dossiers
import sys

sys.path

['/Users/maelioviau-vonfelt/Desktop/Masters/Epitech Data/Cours/M2/Mémoires/Consulting - StudAI/etudIA',
 '/opt/homebrew/Cellar/python@3.12/3.12.11_1/Frameworks/Python.framework/Versions/3.12/lib/python312.zip',
 '/opt/homebrew/Cellar/python@3.12/3.12.11_1/Frameworks/Python.framework/Versions/3.12/lib/python3.12',
 '/opt/homebrew/Cellar/python@3.12/3.12.11_1/Frameworks/Python.framework/Versions/3.12/lib/python3.12/lib-dynload',
 '',
 '/Users/maelioviau-vonfelt/Desktop/Masters/Epitech Data/Cours/M2/Mémoires/Consulting - StudAI/etudIA/env_etudia/lib/python3.12/site-packages']

## 1. Imports et configuration

In [3]:
import pandas as pd
import numpy as np
import re
import pathlib
from pathlib import Path

from src.backend.processing import (
    load_parcoursup_csv,
    drop_unused_columns,
    rename_columns,
    filter_target_year,
    add_is_apprentissage,
    add_internat_code,
    add_amenagement_features
)

## 2. Chargement du dataset brut

In [4]:
csv_path = Path("../data/raw/fr-esr-cartographie_formations_parcoursup.csv")
df_raw = load_parcoursup_csv(csv_path)
df_raw.head()

,Session,Identifiant de l'établissement,Nom de l'établissement,Types d'établissement,Types de formation,Nom long de la formation,Mentions/Spécialités,Formations en apprentissage,Internat,Aménagement,...,Lien vers les données statistiques pour l'année antérieure,Site internet de l'établissement,Localisation,Nom court de la formation,Code interne Parcoursup de la formation,Code interne Parcoursup pour les portails,etablissement_id_paysage,composante_id_paysage,rnd,code_formation
0,2026,0212108C,Polytech Dijon (21),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://data.enseignementsup-recherche.gouv.fr...,http://esirem.u-bourgogne.fr,"47.3122, 5.07398",FI-BAC5,10,7,NaN,NaN,7973,210
1,2026,0492226D,Polytech Angers (49),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://data.enseignementsup-recherche.gouv.fr...,http://www.polytech-angers.fr,"47.4805, -0.59213",FI-BAC5,19,7,5Zmt0,NaN,7231,210
2,2026,0542259M,EEIGM Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://data.enseignementsup-recherche.gouv.fr...,http://eeigm.univ-lorraine.fr/,"48.6954, 6.19324",FI-BAC5,21,7,t6Cq5,9pRyR,9655,210
3,2026,0542307P,ENSGSI Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://data.enseignementsup-recherche.gouv.fr...,https://ensgsi.univ-lorraine.fr/,"48.6949, 6.19356",FI-BAC5,22,7,t6Cq5,Ovw2G,6860,210
4,2026,0597060D,IMT Nord Europe (Villeneuve-d'Ascq - 59),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://data.enseignementsup-recherche.gouv.fr...,https://www.imt-nord-europe.fr,"50.61139, 3.13495",FI-BAC5,24,7,NaN,NaN,21437,210


## 3. Nettoyage et préparation

In [5]:
df_clean = (
    df_raw
    .pipe(drop_unused_columns)
    .pipe(rename_columns)
    .pipe(filter_target_year, 2026)
)

## 4. Vérifications rapides

In [6]:
df_clean.info()
df_clean["session"].value_counts()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   session                       0 non-null      object
 1   id_etablissement              0 non-null      object
 2   name_etablissement            0 non-null      object
 3   type_etablissement            0 non-null      object
 4   type_formation                0 non-null      object
 5   name_formation                0 non-null      object
 6   mentions_specialites          0 non-null      object
 7   apprentissage                 0 non-null      object
 8   internat                      0 non-null      object
 9   amenagement                   0 non-null      object
 10  Informations complémentaires  0 non-null      object
 11  region                        0 non-null      object
 12  departement                   0 non-null      object
 13  commune                       0 non-n

Series([], Name: count, dtype: int64)

In [7]:
df_clean.head()

,session,id_etablissement,name_etablissement,type_etablissement,type_formation,name_formation,mentions_specialites,apprentissage,internat,amenagement,Informations complémentaires,region,departement,commune,link_formation,website_etablissement,short_name_formation


## 5. Test du pipeline complet de nettoyage

In [8]:
from src.backend.processing import clean_parcoursup_data

df_clean = clean_parcoursup_data(csv_path, "2026")
df_clean.head()

,session,id_etablissement,name_etablissement,type_etablissement,type_formation,name_formation,mentions_specialites,apprentissage,internat,amenagement,...,website_etablissement,short_name_formation,is_apprentissage,internat_code,is_presentiel,is_partiel_distance,is_full_distance,has_sport_amenagement,has_artist_amenagement,has_other_amenagement
0,2026,0212108C,Polytech Dijon (21),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://esirem.u-bourgogne.fr,FI-BAC5,0,0,1,0,0,1,1,0
1,2026,0492226D,Polytech Angers (49),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://www.polytech-angers.fr,FI-BAC5,0,0,1,0,0,1,0,0
2,2026,0542259M,EEIGM Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://eeigm.univ-lorraine.fr/,FI-BAC5,0,0,1,0,0,1,1,0
3,2026,0542307P,ENSGSI Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://ensgsi.univ-lorraine.fr/,FI-BAC5,0,0,1,0,0,1,1,1
4,2026,0597060D,IMT Nord Europe (Villeneuve-d'Ascq - 59),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://www.imt-nord-europe.fr,FI-BAC5,0,0,1,0,0,1,1,0


In [9]:
df_clean["session"].unique()

array(['2026'], dtype=object)

In [10]:
df_clean.columns

Index(['session', 'id_etablissement', 'name_etablissement',
       'type_etablissement', 'type_formation', 'name_formation',
       'mentions_specialites', 'apprentissage', 'internat', 'amenagement',
       'Informations complémentaires', 'region', 'departement', 'commune',
       'link_formation', 'website_etablissement', 'short_name_formation',
       'is_apprentissage', 'internat_code', 'is_presentiel',
       'is_partiel_distance', 'is_full_distance', 'has_sport_amenagement',
       'has_artist_amenagement', 'has_other_amenagement'],
      dtype='object')

In [11]:
df_clean.shape

(24058, 25)

# 6. Explorations colonnes catégorielles

## Apprentissage et internat

In [12]:
df_clean["apprentissage"].value_counts(dropna=False)

apprentissage
NaN                            14459
Formations en apprentissage     9599
Name: count, dtype: int64

In [13]:
df_clean["internat"].value_counts(dropna=False)

internat
NaN                                                    23286
Etablissements avec internat pour filles et garçons      747
Etablissements avec internat pour filles                  24
Etablissements avec internat pour garçons                  1
Name: count, dtype: int64

In [14]:
df_clean = (
    df_raw
    .pipe(drop_unused_columns)
    .pipe(rename_columns)
    .pipe(filter_target_year, 2026)
    .pipe(add_is_apprentissage)
    .pipe(add_internat_code)
)

In [15]:
df_clean

,session,id_etablissement,name_etablissement,type_etablissement,type_formation,name_formation,mentions_specialites,apprentissage,internat,amenagement,Informations complémentaires,region,departement,commune,link_formation,website_etablissement,short_name_formation,is_apprentissage,internat_code


In [16]:
from src.backend.processing import clean_parcoursup_data

df_clean = clean_parcoursup_data(csv_path, "2026")
df_clean.head()

,session,id_etablissement,name_etablissement,type_etablissement,type_formation,name_formation,mentions_specialites,apprentissage,internat,amenagement,...,website_etablissement,short_name_formation,is_apprentissage,internat_code,is_presentiel,is_partiel_distance,is_full_distance,has_sport_amenagement,has_artist_amenagement,has_other_amenagement
0,2026,0212108C,Polytech Dijon (21),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://esirem.u-bourgogne.fr,FI-BAC5,0,0,1,0,0,1,1,0
1,2026,0492226D,Polytech Angers (49),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://www.polytech-angers.fr,FI-BAC5,0,0,1,0,0,1,0,0
2,2026,0542259M,EEIGM Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://eeigm.univ-lorraine.fr/,FI-BAC5,0,0,1,0,0,1,1,0
3,2026,0542307P,ENSGSI Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://ensgsi.univ-lorraine.fr/,FI-BAC5,0,0,1,0,0,1,1,1
4,2026,0597060D,IMT Nord Europe (Villeneuve-d'Ascq - 59),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://www.imt-nord-europe.fr,FI-BAC5,0,0,1,0,0,1,1,0


In [17]:
df_clean["is_apprentissage"].value_counts(dropna=False)

is_apprentissage
0    14459
1     9599
Name: count, dtype: int64

## Amenagement

In [18]:
df_clean["amenagement"].value_counts(dropna=False)

amenagement
Enseignement en présentiel                                                                                                                                                                                                                                                                                                         15973
Enseignement en présentiel,Formations avec aménagements pour sportifs haut niveau,Formations avec aménagements pour artistes confirmés,Formations avec aménagements pour autres publics spécifiques                                                                                                                                 3814
Enseignement partiellement à distance                                                                                                                                                                                                                                                                                                908
E

In [19]:
from src.backend.processing import clean_parcoursup_data

df_clean = clean_parcoursup_data(csv_path, "2026")
df_clean.head()

,session,id_etablissement,name_etablissement,type_etablissement,type_formation,name_formation,mentions_specialites,apprentissage,internat,amenagement,...,website_etablissement,short_name_formation,is_apprentissage,internat_code,is_presentiel,is_partiel_distance,is_full_distance,has_sport_amenagement,has_artist_amenagement,has_other_amenagement
0,2026,0212108C,Polytech Dijon (21),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://esirem.u-bourgogne.fr,FI-BAC5,0,0,1,0,0,1,1,0
1,2026,0492226D,Polytech Angers (49),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://www.polytech-angers.fr,FI-BAC5,0,0,1,0,0,1,0,0
2,2026,0542259M,EEIGM Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,http://eeigm.univ-lorraine.fr/,FI-BAC5,0,0,1,0,0,1,1,0
3,2026,0542307P,ENSGSI Nancy - Groupe INP (54),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://ensgsi.univ-lorraine.fr/,FI-BAC5,0,0,1,0,0,1,1,1
4,2026,0597060D,IMT Nord Europe (Villeneuve-d'Ascq - 59),Publics,Formations des écoles d'ingénieurs,Formation d'ingénieur Bac + 5 - Bac général,Formation d'ingénieur Bac + 5,NaN,NaN,"Enseignement en présentiel,Formations avec amé...",...,https://www.imt-nord-europe.fr,FI-BAC5,0,0,1,0,0,1,1,0
